In [5]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.config import CONFIG
from src.utils.io import ensure_dir

In [6]:
def relpath(path: Path) -> str:
    try:
        return str(path.relative_to(CONFIG.root_dir))
    except Exception:
        return str(path.name)


def summarize_dataset(metrics: pd.DataFrame, dataset: str) -> str:
    subset = metrics[metrics["dataset"] == dataset].copy()
    if subset.empty:
        return f"# {dataset.upper()} Results\n\nNo metrics available.\n"

    best = subset.sort_values("roc_auc", ascending=False).iloc[0]

    lines = [
        f"# {dataset.upper()} Results",
        "",
        f"Best model: {best['model_name']}",
        "",
        "Metrics:",
        f"- Accuracy: {best['accuracy']:.4f}",
        f"- Precision: {best['precision']:.4f}",
        f"- Recall: {best['recall']:.4f}",
        f"- F1: {best['f1']:.4f}",
        f"- ROC-AUC: {best['roc_auc']:.4f}",
        "",
    ]
    return "\n".join(lines)

In [7]:
metrics_path = CONFIG.tables_dir / "model_metrics.csv"

if not metrics_path.exists():
    raise FileNotFoundError("No metrics table found. Run the modeling notebooks first.")

metrics = pd.read_csv(metrics_path)

print("Loaded:", relpath(metrics_path))
display(Markdown("### Model metrics overview"))
display(metrics.sort_values(["dataset", "roc_auc"], ascending=[True, False]).reset_index(drop=True))

Loaded: reports/tables/model_metrics.csv


### Model metrics overview

,dataset,model_name,accuracy,precision,recall,f1,roc_auc,is_best
0,endo,logreg,0.609500,0.517241,0.643382,0.573457,0.657465,True
1,endo,xgboost,0.615500,0.541156,0.378676,0.445566,0.638651,False
2,endo,random_forest,0.601000,0.512784,0.442402,0.475000,0.604232,False
3,endo,decision_tree,0.546000,0.449227,0.498775,0.472706,0.538661,False
4,pcos,logreg,0.889908,0.800000,0.888889,0.842105,0.951294,True
5,pcos,xgboost,0.935780,0.967742,0.833333,0.895522,0.950913,False
6,pcos,random_forest,0.926606,0.966667,0.805556,0.878788,0.933029,False
7,pcos,decision_tree,0.889908,0.852941,0.805556,0.828571,0.868531,False


In [8]:
display(Markdown("### Best model per dataset"))

best_by_dataset = (
    metrics.sort_values(["dataset", "roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
)

display(best_by_dataset.reset_index(drop=True))

### Best model per dataset

,dataset,model_name,accuracy,precision,recall,f1,roc_auc,is_best
0,endo,logreg,0.609500,0.517241,0.643382,0.573457,0.657465,True
1,pcos,logreg,0.889908,0.800000,0.888889,0.842105,0.951294,True


In [9]:
summary_dir = CONFIG.root_dir / "reports" / "summary"
ensure_dir(summary_dir)

pcos_summary = summarize_dataset(metrics, "pcos")
endo_summary = summarize_dataset(metrics, "endo")

pcos_md_path = summary_dir / "pcos_results.md"
endo_md_path = summary_dir / "endo_results.md"

pcos_md_path.write_text(pcos_summary, encoding="utf-8")
endo_md_path.write_text(endo_summary, encoding="utf-8")

print("Saved:", relpath(pcos_md_path))
print("Saved:", relpath(endo_md_path))

Saved: reports/summary/pcos_results.md
Saved: reports/summary/endo_results.md


In [10]:
display(Markdown("### PCOS summary"))
display(Markdown(pcos_summary))

display(Markdown("### Endometriosis summary"))
display(Markdown(endo_summary))

### PCOS summary

# PCOS Results

Best model: logreg

Metrics:
- Accuracy: 0.8899
- Precision: 0.8000
- Recall: 0.8889
- F1: 0.8421
- ROC-AUC: 0.9513


### Endometriosis summary

# ENDO Results

Best model: logreg

Metrics:
- Accuracy: 0.6095
- Precision: 0.5172
- Recall: 0.6434
- F1: 0.5735
- ROC-AUC: 0.6575


In [11]:
display(Markdown("### Interpretation"))

for _, row in best_by_dataset.iterrows():
    print(
        f"{row['dataset'].upper()}: best model = {row['model_name']} | "
        f"accuracy = {row['accuracy']:.4f}, f1 = {row['f1']:.4f}, roc_auc = {row['roc_auc']:.4f}"
    )

### Interpretation

ENDO: best model = logreg | accuracy = 0.6095, f1 = 0.5735, roc_auc = 0.6575
PCOS: best model = logreg | accuracy = 0.8899, f1 = 0.8421, roc_auc = 0.9513
